In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="FR",
    max_iter=10000,
    tol=1e-100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.7002364993095398
epoch:  1, loss: 0.37124404311180115
epoch:  2, loss: 0.22621025145053864
epoch:  3, loss: 0.14423760771751404
epoch:  4, loss: 0.09647757560014725
epoch:  5, loss: 0.06922487169504166
epoch:  6, loss: 0.054031237959861755
epoch:  7, loss: 0.04574641212821007
epoch:  8, loss: 0.041315361857414246
epoch:  9, loss: 0.038978658616542816
epoch:  10, loss: 0.03775857388973236
epoch:  11, loss: 0.03712454438209534
epoch:  12, loss: 0.036794260144233704
epoch:  13, loss: 0.03662038967013359
epoch:  14, loss: 0.03652698174118996
epoch:  15, loss: 0.03647422790527344
epoch:  16, loss: 0.03644227609038353
epoch:  17, loss: 0.03639208897948265
epoch:  18, loss: 0.03634501248598099
epoch:  19, loss: 0.03631714731454849
epoch:  20, loss: 0.03628665581345558
epoch:  21, loss: 0.03624533489346504
epoch:  22, loss: 0.03622087463736534
epoch:  23, loss: 0.03619666397571564
epoch:  24, loss: 0.03615884855389595
epoch:  25, loss: 0.03613649308681488
epoch:  26, loss: 0

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7719761140428013
Test metrics:  R2 = 0.732989943295256


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.15540561079978943


epoch:  1, loss: 0.10111291706562042
epoch:  2, loss: 0.07181117683649063
epoch:  3, loss: 0.0556507408618927
epoch:  4, loss: 0.04665394127368927
epoch:  5, loss: 0.04164755716919899
epoch:  6, loss: 0.03886064514517784
epoch:  7, loss: 0.03730734810233116
epoch:  8, loss: 0.03644118830561638
epoch:  9, loss: 0.03595738112926483
epoch:  10, loss: 0.03568606078624725
epoch:  11, loss: 0.03553349897265434
epoch:  12, loss: 0.0354468934237957
epoch:  13, loss: 0.03539697080850601
epoch:  14, loss: 0.035367708653211594
epoch:  15, loss: 0.03534993901848793
epoch:  16, loss: 0.035340774804353714
epoch:  17, loss: 0.03531850129365921
epoch:  18, loss: 0.03530462831258774
epoch:  19, loss: 0.035291753709316254
epoch:  20, loss: 0.035274527966976166
epoch:  21, loss: 0.03526951000094414
epoch:  22, loss: 0.03524751961231232
epoch:  23, loss: 0.03523403778672218
epoch:  24, loss: 0.035223498940467834
epoch:  25, loss: 0.03520641103386879
epoch:  26, loss: 0.035203319042921066
epoch:  27, loss:

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7288705682390151
Test metrics:  R2 = 0.7049251105645685


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PRP+"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.33226534724235535
epoch:  1, loss: 0.20020116865634918
epoch:  2, loss: 0.1298598349094391
epoch:  3, loss: 0.09024104475975037
epoch:  4, loss: 0.06720997393131256
epoch:  5, loss: 0.05381093919277191
epoch:  6, loss: 0.04603910818696022
epoch:  7, loss: 0.04156408831477165
epoch:  8, loss: 0.03900410234928131
epoch:  9, loss: 0.0375477634370327
epoch:  10, loss: 0.03672301396727562
epoch:  11, loss: 0.03625737503170967
epoch:  12, loss: 0.03599497675895691
epoch:  13, loss: 0.03584716096520424
epoch:  14, loss: 0.0357636883854866
epoch:  15, loss: 0.035716187208890915
epoch:  16, loss: 0.0356888622045517
epoch:  17, loss: 0.03567281737923622
epoch:  18, loss: 0.03566303849220276
epoch:  19, loss: 0.035658933222293854
epoch:  20, loss: 0.03564665839076042
epoch:  21, loss: 0.03563903272151947
epoch:  22, loss: 0.03563089668750763
epoch:  23, loss: 0.035621315240859985
epoch:  24, loss: 0.035619962960481644
epoch:  25, loss: 0.03560791537165642
epoch:  26, loss: 0.03

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7254274669914568
Test metrics:  R2 = 0.7077610343199565


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="HS"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.04450661689043045
epoch:  1, loss: 0.040000926703214645
epoch:  2, loss: 0.03792372718453407
epoch:  3, loss: 0.03697522357106209
epoch:  4, loss: 0.03654453530907631
epoch:  5, loss: 0.036349210888147354
epoch:  6, loss: 0.03626013919711113
epoch:  7, loss: 0.036218829452991486
epoch:  8, loss: 0.036198925226926804
epoch:  9, loss: 0.03618859499692917
epoch:  10, loss: 0.03618253394961357
epoch:  11, loss: 0.03616863861680031
epoch:  12, loss: 0.036157846450805664
epoch:  13, loss: 0.03615165129303932
epoch:  14, loss: 0.036139730364084244
epoch:  15, loss: 0.03612852841615677
epoch:  16, loss: 0.03612218797206879
epoch:  17, loss: 0.03611237183213234
epoch:  18, loss: 0.03610065206885338
epoch:  19, loss: 0.03609415888786316
epoch:  20, loss: 0.03608623519539833
epoch:  21, loss: 0.03607403486967087
epoch:  22, loss: 0.03606734797358513
epoch:  23, loss: 0.03606118634343147
epoch:  24, loss: 0.036048416048288345
epoch:  25, loss: 0.03604155406355858
epoch:  26, los

In [12]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7584686387858139
Test metrics:  R2 = 0.7257513594847944


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="DY"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.6889912486076355
epoch:  1, loss: 0.39094412326812744
epoch:  2, loss: 0.22775958478450775
epoch:  3, loss: 0.138099804520607
epoch:  4, loss: 0.08913301676511765
epoch:  5, loss: 0.06288762390613556
epoch:  6, loss: 0.049263618886470795
epoch:  7, loss: 0.04239954426884651
epoch:  8, loss: 0.03902452066540718
epoch:  9, loss: 0.037395209074020386
epoch:  10, loss: 0.03661835938692093
epoch:  11, loss: 0.03625018522143364
epoch:  12, loss: 0.036075495183467865
epoch:  13, loss: 0.03599144518375397
epoch:  14, loss: 0.03594960644841194
epoch:  15, loss: 0.035927485674619675
epoch:  16, loss: 0.03591442108154297
epoch:  17, loss: 0.03588709235191345
epoch:  18, loss: 0.03586446866393089
epoch:  19, loss: 0.03585130348801613
epoch:  20, loss: 0.03582913801074028
epoch:  21, loss: 0.03580591827630997
epoch:  22, loss: 0.03579289838671684
epoch:  23, loss: 0.03577877953648567
epoch:  24, loss: 0.03575456514954567
epoch:  25, loss: 0.03574131429195404
epoch:  26, loss: 0.0

In [14]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7654174711652609
Test metrics:  R2 = 0.7194352249811926
